# Atividade Prática - Parte 2
### Grupo 2:<br>
Victor Monteiro <br>
Arthur Horta <br>
João Henrique <br>
Pedro Fioravante <br>


## Tarefa 1: Importe os dados para este notebook e gere um modelo de regressão múltipla **com pelo menos** 2 variáveis explicativas.

In [ ]:
# Importação de bibliotecas
import pandas as pd, numpy as np
import statsmodels.api as sm
from pathlib import Path

# Leitura do arquivo CSV local e conversão da coluna 'date'
df = pd.read_csv(Path("Dados") / "Valores Variaveis Interpolado.csv")
df['date'] = pd.to_datetime(df['date'])

# Cálculo dos retornos logarítmicos (exceto a coluna 'date')
df_ret = np.log(df.drop(columns='date') / df.drop(columns='date').shift(1)).dropna()
df_ret['date'] = df['date'].iloc[1:]  # Reposiciona as datas

# Define variável dependente (petróleo) e independentes (demais colunas)
y = df_ret['price_crude_oil']
X = sm.add_constant(df_ret.drop(columns=['price_crude_oil', 'date']))

# Ajusta o modelo de regressão OLS (MQO)
modelo1 = sm.OLS(y, X).fit()

# Exibe o resumo dos resultados da regressão
print(modelo1.summary())

                            OLS Regression Results                            
Dep. Variable:        price_crude_oil   R-squared:                       0.888
Model:                            OLS   Adj. R-squared:                  0.887
Method:                 Least Squares   F-statistic:                     1269.
Date:                Wed, 14 May 2025   Prob (F-statistic):               0.00
Time:                        21:53:45   Log-Likelihood:                 8394.7
No. Observations:                2258   AIC:                        -1.676e+04
Df Residuals:                    2243   BIC:                        -1.667e+04
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     0.00

## Tarefa 2: Estime agora o modelo linear  $$y = \alpha + \beta_1 x_1 + \beta_2 x_1^2 + \beta_3x_2 + u$$ e apresente os resultados. Escolha uma variável que faça sentido para gerar o termo quadrático. Há algum ponto de máximo ou mínimo identificado? Qual o valor estimado?

In [ ]:
# Cria o termo quadrático do índice do dólar
df_ret['usd_index_sqr'] = df_ret['price_us dollar_index'] ** 2

# Define a variável dependente (retorno do petróleo) e as explicativas (incluindo o termo quadrático)
y = df_ret['price_crude_oil']
X = sm.add_constant(df_ret.drop(columns=['price_crude_oil', 'date']))
modelo2 = sm.OLS(y, X).fit()

# Calcula o ponto de máximo ou mínimo da parábola: x* = -β₁ / (2 * β₂)
b1, b2 = modelo2.params['price_us dollar_index'], modelo2.params['usd_index_sqr']
x_star = -b1 / (2 * b2) if b2 != 0 else None
tipo = "mínimo" if b2 > 0 else "máximo" if b2 < 0 else "indefinido"

# Exibe o resumo do modelo e o ponto de extremum (se existir)
print(modelo2.summary())
print(f"\n📈 Termo quadrático: usd_index_sqr")
print(f"• β₁: {b1:.4f} | β₂: {b2:.4f}")
print(f"• Ponto de {tipo}: índice do dólar ≈ {x_star:.4f}" if x_star else "• β₂ = 0 → sem ponto extremo definido")

                            OLS Regression Results                            
Dep. Variable:        price_crude_oil   R-squared:                       0.888
Model:                            OLS   Adj. R-squared:                  0.887
Method:                 Least Squares   F-statistic:                     1185.
Date:                Wed, 14 May 2025   Prob (F-statistic):               0.00
Time:                        21:53:49   Log-Likelihood:                 8395.6
No. Observations:                2258   AIC:                        -1.676e+04
Df Residuals:                    2242   BIC:                        -1.667e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     0.00

## Tarefa 3:  Agora, crie ou use uma variável dummy na sua base de dados, e incorpore esta variável no modelo estimado na Tarefa 1 e apresente os resultados.

In [ ]:
# Cria uma variável dummy que indica o período de choque no preço do petróleo (anos 2014 a 2016)
df_ret['price_shock'] = df_ret['date'].apply(lambda x: 1 if 2014 <= x.year <= 2016 else 0)

# Define a variável dependente (retorno do petróleo) e as independentes (incluindo a dummy)
y = df_ret['price_crude_oil']
X = sm.add_constant(df_ret.drop(columns=['price_crude_oil', 'date']))

# Ajusta o modelo de regressão OLS com a dummy de choque e exibe os resultados
modelo3 = sm.OLS(y, X).fit()
print(modelo3.summary())

                            OLS Regression Results                            
Dep. Variable:        price_crude_oil   R-squared:                       0.888
Model:                            OLS   Adj. R-squared:                  0.887
Method:                 Least Squares   F-statistic:                     1112.
Date:                Wed, 14 May 2025   Prob (F-statistic):               0.00
Time:                        21:53:53   Log-Likelihood:                 8397.3
No. Observations:                2258   AIC:                        -1.676e+04
Df Residuals:                    2241   BIC:                        -1.666e+04
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                  8.473e-

## Tarefa 4: Por fim, faça os teses de Heterocedasticidade e de Correlação Serial para os modelos estimados nas tarefas 1, 2 e 3.

Teste de Breusch-Pagan e Plotagem dos Resíduos

In [24]:
# Modelos e nomes
modelos = [modelo1, modelo2, modelo3]
nomes = ['Modelo_1', 'Modelo_2', 'Modelo_3']
colunas = ["Teste", "Estatística", "Valor-p", "Estatística F", "Valor-p F"]

for modelo, nome in zip(modelos, nomes):
    path = Path("Saídas") / nome
    path.mkdir(parents=True, exist_ok=True)

# Salva resumos robustos
for tipo, fname in [('HC0', 'resumo_white.txt'), ('HAC', 'resumo_neweywest.txt')]:
    if tipo == 'HAC':
        resumo = modelo.get_robustcov_results(cov_type=tipo, maxlags=6)
    else:
        resumo = modelo.get_robustcov_results(cov_type=tipo)
    with open(path / fname, "w") as f:
        f.write(resumo.summary().as_text())

    # Testes estatísticos
    bp = sms.het_breuschpagan(modelo.resid, modelo.model.exog)
    bg = smd.acorr_breusch_godfrey(modelo, nlags=2)
    dw = sms.durbin_watson(modelo.resid)

    # Gráfico dos resíduos
    plt.figure(figsize=(8, 4))
    plt.plot(df_ret['date'], modelo.resid)
    plt.title(f'Resíduos — {nome}')
    plt.xlabel('Data'); plt.ylabel('Resíduos'); plt.grid(True)
    plt.tight_layout(); plt.savefig(path / "residuos.png"); plt.close()

    # Tabela dos testes
    tabela = pd.DataFrame([
        ["Breusch-Pagan", *bp],
        ["Durbin-Watson", dw, None, None, None],
        ["Breusch-Godfrey", *bg]
    ], columns=colunas)

    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axis('off')
    tbl = pd.plotting.table(ax, tabela.round(4), loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.2, 1.2)
    plt.savefig(path / "tabela_testes.png", bbox_inches='tight'); plt.close()

print("✅ Todas as saídas e testes foram salvos nas pastas dos modelos.")

✅ Todas as saídas e testes foram salvos nas pastas dos modelos.
